# 常用标准库 · 知识点

## collections

### 1、namedtuple 具名元组

In [ ]:
from collections import namedtuple

# namedtuple(类型名, 字段名列表)：生成一个轻量级"类"，跟普通 tuple 一样不可变，但能用属性名访问字段
# 对比第05章的 dataclass：namedtuple 更轻，没有方法/默认值这些功能，适合纯数据的小对象
Point = namedtuple("Point", ["x", "y"])
p = Point(1, 2)
print(p.x, p.y)     # 1 2，按属性名访问
print(p[0], p[1])   # 1 2，也能像普通 tuple 一样按下标访问
print(p)             # Point(x=1, y=2)

# _replace()：基于现有实例创建一个改了某些字段的新实例（namedtuple 不可变，不能直接 p.x = 10）
p2 = p._replace(x=10)
print(p2)            # Point(x=10, y=2)

### 2、Counter 计数器

In [ ]:
from collections import Counter

# Counter：专门用来计数的字典子类，key 是元素，value 是出现次数
words = ["a", "b", "a", "c", "b", "a"]
c = Counter(words)
print(c)                  # Counter({'a': 3, 'b': 2, 'c': 1})
print(c["a"])              # 3
print(c["not_exist"])      # 0，跟普通 dict 不一样，访问不存在的 key 不会报 KeyError，直接返回 0

print(c.most_common(2))    # [('a', 3), ('b', 2)]  出现次数最多的 2 个
c.update(["a", "d"])        # 累加计数，而不是覆盖
print(c)

### 3、defaultdict 默认值字典

In [ ]:
from collections import defaultdict

# defaultdict(工厂函数)：访问不存在的 key 时，自动用工厂函数生成一个默认值并赋进去，不会报 KeyError
# 常见场景：按 key 分组收集数据，不用每次先判断 key 在不在
d = defaultdict(list)
pairs = [("a", 1), ("b", 2), ("a", 3)]
for k, v in pairs:
    d[k].append(v)    # 不用先写 if k not in d: d[k] = []
print(d)               # defaultdict(<class 'list'>, {'a': [1, 3], 'b': [2]})
print(dict(d))          # 转成普通 dict 打印更干净：{'a': [1, 3], 'b': [2]}

### 4、deque 双端队列

In [ ]:
from collections import deque

# deque（双端队列）：两端插入/删除都是 O(1)；普通 list 在开头插入/删除是 O(n)，数据量大时差距明显
dq = deque([1, 2, 3])
dq.append(4)        # 右边添加
dq.appendleft(0)     # 左边添加
print(dq)            # deque([0, 1, 2, 3, 4])
dq.pop()              # 右边弹出
dq.popleft()           # 左边弹出
print(dq)              # deque([1, 2, 3])

# maxlen 参数：固定长度的"滑动窗口"，满了之后继续添加，最早的元素会从另一端自动挤出去
window = deque(maxlen=3)
for i in range(5):
    window.append(i)
    print(window)
# deque([0], maxlen=3) -> deque([0, 1], maxlen=3) -> deque([0, 1, 2], maxlen=3)
# -> deque([1, 2, 3], maxlen=3) -> deque([2, 3, 4], maxlen=3)

## itertools

### 5、chain 与 product

In [ ]:
from itertools import chain, product

# chain(*iterables)：把多个可迭代对象首尾接起来当成一个整体迭代，不需要先合并成一个新列表
a = [1, 2]
b = [3, 4]
for x in chain(a, b):
    print(x)            # 1 2 3 4

# product(*iterables)：笛卡尔积，等价于嵌套 for 循环，但写法更简洁
for combo in product([1, 2], ["a", "b"]):
    print(combo)          # (1, 'a') (1, 'b') (2, 'a') (2, 'b')

### 6、permutations 与 combinations

In [ ]:
from itertools import permutations, combinations

# permutations(iterable, r)：排列，元素顺序不同算不同结果；r 不写默认是序列全长度
print(list(permutations([1, 2, 3], 2)))
# [(1, 2), (1, 3), (2, 1), (2, 3), (3, 1), (3, 2)]

# combinations(iterable, r)：组合，只看选了哪些元素，不管顺序，所以数量比 permutations 少
print(list(combinations([1, 2, 3], 2)))
# [(1, 2), (1, 3), (2, 3)]

### 7、groupby 分组

In [ ]:
from itertools import groupby

# groupby(iterable, key=None)：把"连续相同 key 的元素"分到一组
# 注意：groupby 只看相邻元素，用之前必须先按同样的 key 排序，否则相同 key 隔开了不会被分到一组
data = [("a", 1), ("a", 2), ("b", 3), ("b", 4), ("a", 5)]
data.sort(key=lambda x: x[0])   # 排序后同 key 的项才会挨在一起
for key, group in groupby(data, key=lambda x: x[0]):
    print(key, list(group))
# a [('a', 1), ('a', 2), ('a', 5)]
# b [('b', 3), ('b', 4)]

### 8、count / cycle / islice

In [ ]:
from itertools import count, cycle, islice

# count(start, step)：从 start 开始按 step 无限递增，自己不会停
# cycle(iterable)：把一个序列循环无限次
# 这两个都是无限迭代器，必须配合 islice（或 break）限制次数，否则会死循环
print(list(islice(count(10, 2), 5)))        # [10, 12, 14, 16, 18]
print(list(islice(cycle(["a", "b"]), 5)))   # ['a', 'b', 'a', 'b', 'a']

## functools

### 9、reduce 累积计算

In [ ]:
from functools import reduce

# reduce(函数, 可迭代对象, 初始值=可选)：把"前一步结果"和"下一个元素"反复传给函数，最后剩一个值
# 等价于一个累积计算的 for 循环；不写初始值时，第一次调用用序列的前两个元素
nums = [1, 2, 3, 4]
total = reduce(lambda acc, x: acc + x, nums)
print(total)        # 10，等价于 ((1+2)+3)+4

product_with_init = reduce(lambda acc, x: acc * x, nums, 1)   # 给个初始值 1，避免空列表时报错
print(product_with_init)   # 24

### 10、partial 固定参数

In [ ]:
from functools import partial

# partial(函数, 固定的参数)：预先把某些参数"焊死"，返回一个新函数，调用时只需要补剩下的参数
def power(base, exponent):
    return base ** exponent

square = partial(power, exponent=2)   # 固定 exponent=2，square 现在只需要传 base
print(square(5))     # 25
print(square(10))    # 100

### 11、lru_cache 缓存装饰器

In [ ]:
from functools import lru_cache

# @lru_cache(maxsize=...)：给函数加缓存，相同参数第二次调用直接返回缓存结果，不再重复计算
# 常用在递归（比如斐波那契数列）或开销大的纯函数上；maxsize 控制最多缓存多少种不同参数的结果，None 表示不限制
@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(30))             # 很快，因为重复子问题被缓存了，不是每次都重新算
print(fib.cache_info())    # 查看缓存命中情况：hits/misses/maxsize/currsize

### 12、wraps 保留函数元信息

In [ ]:
from functools import wraps

# 写装饰器时，不加 @wraps 会导致被装饰函数的 __name__/__doc__ 等元信息丢失，变成内部 wrapper 的信息
def log_call(func):
    @wraps(func)              # 把 func 的元信息"拷贝"给 wrapper，让外界看到的还是原函数的信息
    def wrapper(*args, **kwargs):
        print(f"调用：{func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log_call
def greet(name):
    """打个招呼"""
    return f"Hello, {name}"

print(greet("Tom"))
print(greet.__name__)   # greet，不加 @wraps 的话这里会变成 'wrapper'
print(greet.__doc__)    # 打个招呼

## datetime

### 13、datetime 基础

In [ ]:
from datetime import datetime, date

# datetime.now()：当前日期时间；date.today()：只要日期，没有时间部分
now = datetime.now()
print(now)                       # 2026-xx-xx xx:xx:xx.xxxxxx
print(now.year, now.month, now.day, now.hour, now.minute)

# 也可以直接构造指定的时间点
d = datetime(2024, 1, 1, 12, 30)
print(d)                          # 2024-01-01 12:30:00

print(date.today())               # 只有日期，没有时间部分

### 14、timedelta 时间运算

In [ ]:
from datetime import datetime, timedelta

# timedelta：表示一段时间长度，可以直接跟 datetime 做加减
now = datetime(2024, 1, 1)
tomorrow = now + timedelta(days=1)
print(tomorrow)                # 2024-01-02 00:00:00

three_hours_later = now + timedelta(hours=3)
print(three_hours_later)        # 2024-01-01 03:00:00

# 两个 datetime 相减，结果就是一个 timedelta
diff = tomorrow - now
print(diff)                      # 1 day, 0:00:00
print(diff.days, diff.seconds)   # 1 0

### 15、格式化与解析：strftime / strptime

In [ ]:
from datetime import datetime

# strftime（string format time）：把 datetime 对象格式化成指定样式的字符串
now = datetime(2024, 3, 15, 9, 5)
print(now.strftime("%Y-%m-%d %H:%M:%S"))   # 2024-03-15 09:05:00
print(now.strftime("%Y年%m月%d日"))          # 2024年03月15日
# 常用占位符：%Y 四位年 / %m 两位月 / %d 两位日 / %H 24小时制 / %M 分 / %S 秒

# strptime（string parse time）：反过来，把字符串解析成 datetime 对象，格式必须跟字符串完全对应
parsed = datetime.strptime("2024-03-15 09:05:00", "%Y-%m-%d %H:%M:%S")
print(parsed)          # 2024-03-15 09:05:00
print(type(parsed))    # <class 'datetime.datetime'>

## re（正则表达式）

### 16、match / search 基础匹配

In [ ]:
import re

# re.match(pattern, string)：只从字符串开头匹配，开头对不上直接返回 None
# re.search(pattern, string)：在整个字符串里找第一个匹配的位置，不要求从开头开始
text = "order id: 12345"
print(re.match(r"order", text))    # 匹配成功，从开头开始
print(re.match(r"id", text))        # None，因为开头不是 id

m = re.search(r"\d+", text)         # \d+ 表示一个或多个数字
print(m)                            # <re.Match object; span=(10, 15), match='12345'>
print(m.group())                    # 12345，取出匹配到的内容

### 17、findall / finditer 查找所有匹配

In [ ]:
import re

text = "phone: 138-0000-1111, 159-2222-3333"

# findall：返回所有匹配到的字符串组成的 list（没有分组时）
print(re.findall(r"\d{3}-\d{4}-\d{4}", text))
# ['138-0000-1111', '159-2222-3333']

# finditer：返回一个迭代器，每个元素是 Match 对象，能拿到位置信息，数据量大或需要位置时用这个
for m in re.finditer(r"\d{3}-\d{4}-\d{4}", text):
    print(m.group(), m.span())   # span 是 (起始位置, 结束位置)

### 18、sub 替换

In [ ]:
import re

# re.sub(pattern, repl, string, count=0)：把匹配到的内容替换成 repl
# count 不写默认全部替换，写了具体数字就只替换前面几个
text = "联系电话：13800001111"
masked = re.sub(r"\d{4}(?=\d{4}$)", "****", text)   # (?=...) 是前瞻，只用来判断不消耗字符，给中间4位打码
print(masked)          # 联系电话：138****1111

# repl 也可以是一个函数，对每个匹配项动态生成替换内容
result = re.sub(r"\d+", lambda m: str(int(m.group()) * 2), "a1 b2 c3")
print(result)           # a2 b4 c6

### 19、分组与 compile

In [ ]:
import re

# 圆括号 () 是分组，可以单独取出某一部分；(?P<name>...) 是命名分组，用名字取比用序号取更清晰
text = "2024-03-15"
m = re.match(r"(?P<year>\d{4})-(?P<month>\d{2})-(?P<day>\d{2})", text)
print(m.group(0))                     # 2024-03-15，整体匹配
print(m.group(1), m.group("year"))    # 2024 2024，序号和名字两种取法
print(m.groupdict())                   # {'year': '2024', 'month': '03', 'day': '15'}

# re.compile(pattern)：预编译正则得到一个 Pattern 对象，反复用同一个正则匹配多次时能省去重复编译的开销
pattern = re.compile(r"\d+")
print(pattern.findall("a1 b22 c333"))   # ['1', '22', '333']

## logging

### 20、logging 基础用法

In [ ]:
import logging

# logging 比 print 更适合做日志：可以分级别、可以同时输出到多个地方、可以控制格式
# 5 个常用级别（从低到高）：DEBUG < INFO < WARNING < ERROR < CRITICAL
# basicConfig 设置全局最低输出级别，低于这个级别的日志会被直接忽略
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

logging.debug("调试信息")     # 不会输出，因为级别比 INFO 低
logging.info("普通信息")       # 会输出
logging.warning("警告信息")     # 会输出
logging.error("错误信息")       # 会输出

### 21、Handler 与 Formatter

In [ ]:
import logging

# 实际项目里很少直接用 root logger（上面那种 logging.xxx() 写法），通常用 getLogger() 拿一个具名 logger
# Handler 决定"日志往哪里输出"（控制台/文件/网络...），Formatter 决定"输出格式"
logger = logging.getLogger("my_app")
logger.setLevel(logging.DEBUG)
logger.propagate = False
# propagate 默认是 True：日志会同时"冒泡"传给 root logger 处理一遍
# 如果上面（第20节）已经跑过 basicConfig，root logger 也有自己的 handler，
# 不关掉 propagate 的话，同一条日志会被 my_app 自己的 handler 和 root 的 handler 各打印一次，看起来像重复打印了两遍

formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")

console_handler = logging.StreamHandler()        # 输出到控制台
console_handler.setFormatter(formatter)

file_handler = logging.FileHandler("app.log", encoding="utf-8")   # 输出到文件
file_handler.setFormatter(formatter)

logger.addHandler(console_handler)
logger.addHandler(file_handler)

logger.info("这条日志同时出现在控制台和 app.log 文件里")